# Q5: 可疑活动地点分析与证据链构建

**VAST Challenge 2021 MC2 — Kronos 事件调查**

本 Notebook 综合 Q1-Q4 的分析成果，从四个维度对可疑活动地点进行评分和排序:
- **Vector A (经济异常)**: 高价工业采购、$10,000 极端交易、CC-Loyalty 系统性偏移
- **Vector B (时间碰撞)**: 凌晨交易 (0-5点)、深夜 GPS 共现事件
- **Vector C (POK 监视)**: 未分配车辆 (101/104/105/106/107) 的持久停留模式
- **Vector D (网络裂隙)**: 非工作时段跨部门共现、异常社交模式

最终输出 1-10 处可疑活动地点，附完整证据链与执法建议。

## 0. 环境设置与数据加载

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from pathlib import Path
from math import pi
import warnings
warnings.filterwarnings("ignore")

# 全局样式
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "savefig.facecolor": "white",
})

DATA_DIR = Path("../data/processed")
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 配色
C = {
    "danger": "#e02424", "warning": "#f59e0b", "primary": "#1a56db",
    "industrial": "#b434eb", "night": "#7c3aed", "unassigned": "#e02424",
    "engineer": "#3b82f6", "executive": "#8b5cf6", "security": "#ef4444",
    "facilities": "#f59e0b", "it": "#06b6d4", "transport": "#f97316",
    "retail": "#0e9f6e",
}

print("✅ 环境就绪")

In [ ]:
# 加载所有预处理数据层
anomaly    = pd.read_csv(DATA_DIR / "anomaly_transactions.csv")
txn_long   = pd.read_csv(DATA_DIR / "transactions_long.csv")
card_own   = pd.read_csv(DATA_DIR / "card_ownership.csv")
cc_lm      = pd.read_csv(DATA_DIR / "cc_loyalty_matched.csv")
loc_cat    = pd.read_csv(DATA_DIR / "location_category.csv")
cooc       = pd.read_csv(DATA_DIR / "q4_cooccurrence_details.csv")
net_edges  = pd.read_csv(DATA_DIR / "q4_network_edges.csv")
centrality = pd.read_csv(DATA_DIR / "q4_centrality_stats.csv")
community  = pd.read_csv(DATA_DIR / "q4_community_assignments.csv")
unassigned_daily = pd.read_csv(DATA_DIR / "unassigned_vehicle_daily_summary.csv")
unassigned_hourly = pd.read_csv(DATA_DIR / "unassigned_vehicle_stop_hourly.csv")
hotspots   = pd.read_csv(DATA_DIR / "gps_stop_hotspots_points.csv")

# 日期解析
for df in [anomaly, txn_long, cooc, unassigned_daily]:
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
anomaly["timestamp"] = pd.to_datetime(anomaly["timestamp"])

print(f"异常交易: {len(anomaly)} | 交易长表: {len(txn_long)} | 卡片归属: {len(card_own)}")
print(f"CC-Loyalty匹配: {len(cc_lm)} | 共现事件: {len(cooc)} | 网络边: {len(net_edges)}")
print(f"GPS热点: {len(hotspots)} | 未分配车辆记录: {len(unassigned_daily)}")
print("✅ 全部数据加载完成")

## 1. Vector A: 经济异常分析

检查高价工业采购、$10,000 极端交易、CC-Loyalty 系统性价格偏移。

In [ ]:
# A1: 工业地点高价交易概况
industrial_high = anomaly[anomaly["is_industrial_location"] == True]
extreme_txn = anomaly[anomaly["is_extreme_price"] == True]

print("=" * 60)
print("Vector A: 经济异常")
print("=" * 60)
print(f"\n工业高价交易: {len(industrial_high)} 笔")
print(f"极端价格交易: {len(extreme_txn)} 笔")

# 极端交易详情
print("\n🔴 极端交易 ($10,000):")
for _, row in extreme_txn.iterrows():
    card_info = card_own[card_own["card_id"] == row["card_id"]]
    owner = card_info.iloc[0]["primary_employee"] if len(card_info) > 0 else "?"
    dept = card_info.iloc[0]["primary_department"] if len(card_info) > 0 else "?"
    print(f"  日期: {row['date'].strftime('%Y-%m-%d')} | 地点: {row['location_clean']}")
    print(f"  金额: ${row['price']:,.2f} | 卡片: {row['card_id']} | 推断持有者: {owner} ({dept})")
    print(f"  异常原因: {row['anomaly_reason']}")

# 按地点聚合经济异常评分
econ_scores = {}
for loc in anomaly["location_clean"].unique():
    loc_df = anomaly[anomaly["location_clean"] == loc]
    score = loc_df["is_high_price"].sum() * 2 + loc_df["is_extreme_price"].sum() * 10 + loc_df["is_industrial_location"].sum() * 1
    econ_scores[loc] = score

# 加入交易总额
loc_total_spend = txn_long.groupby("location_clean")["price"].sum()
for loc in econ_scores:
    if loc in loc_total_spend.index:
        econ_scores[loc] += np.log1p(loc_total_spend[loc]) * 0.5

econ_df = pd.DataFrame(list(econ_scores.items()), columns=["location", "economic_score"]).sort_values("economic_score", ascending=False)
print(f"\n经济异常评分 Top 5:")
print(econ_df.head(5).to_string(index=False))

In [ ]:
# A2: CC-Loyalty 系统性价格偏移 ($20/$60/$80 固定差异)
cc_lm_copy = cc_lm.copy()
cc_lm_copy["price_diff"] = cc_lm_copy["cc_price"] - cc_lm_copy["loyalty_price"]

print("CC-Loyalty 匹配价格差异分析")
print("-" * 50)
print(f"精确匹配 ($0差异): {(cc_lm_copy['price_diff'].abs() < 0.01).sum():,} 对")
print(f"系统性 $20 偏移: {(abs(cc_lm_copy['price_diff'] - 20) < 1).sum()} 对")
print(f"系统性 $60 偏移: {(abs(cc_lm_copy['price_diff'] - 60) < 1).sum()} 对")
print(f"系统性 $80 偏移: {(abs(cc_lm_copy['price_diff'] - 80) < 1).sum()} 对")
print(f"\n⚠️ 所有系统性偏移均为 CC 价格 > Loyalty 价格")
print("这强烈暗示 CC 交易数据可能被人为抬高，存在数据篡改嫌疑")

## 2. Vector B: 时间碰撞分析

识别凌晨 0-5 点的交易和 GPS 共现事件。

In [ ]:
# B1: 凌晨交易
early_morning = anomaly[anomaly["is_early_morning"] == True]
print("=" * 60)
print("Vector B: 时间碰撞")
print("=" * 60)
print(f"\n🌙 凌晨交易 (0-5点): {len(early_morning)} 笔")
print("全部发生在 Kronos Mart, 全部在凌晨 3:00 整:")
for _, row in early_morning.iterrows():
    print(f"  {row['date'].strftime('%Y-%m-%d')} 03:00 | {row['location_clean']} | ${row['price']:.2f} | {row['card_id']}")

# B2: 夜间 GPS 共现
night_cooc = cooc[cooc["is_night"] == True]
print(f"\n🌙 夜间 GPS 共现 (0-5点): {len(night_cooc)} 次")
for _, row in night_cooc.iterrows():
    print(f"  {row['date'].strftime('%Y-%m-%d')} {int(row['start_hour']):02d}:00 | "
          f"{row['emp_a']} ↔ {row['emp_b']} | "
          f"({row['mean_lat']:.4f}, {row['mean_lon']:.4f}) | "
          f"重叠 {row['overlap_min']:.1f} 分钟")

# 时间碰撞评分
temporal_scores = {}
for loc in early_morning["location_clean"].unique():
    temporal_scores[loc] = temporal_scores.get(loc, 0) + 5
for _, row in night_cooc.iterrows():
    coord_key = f"({row['mean_lat']:.3f}, {row['mean_lon']:.3f})"
    temporal_scores[coord_key] = temporal_scores.get(coord_key, 0) + 3

print(f"\n时间碰撞评分: {len(temporal_scores)} 个位置")

## 3. Vector C: POK 监视足迹

分析 5 辆未分配车辆 (101/104/105/106/107) 的活动模式。

In [ ]:
# C: POK 监视 — 未分配车辆
unassigned_ids = ["101", "104", "105", "106", "107"]

print("=" * 60)
print("Vector C: POK 监视足迹")
print("=" * 60)

# 未分配车辆活动概况
for vid in unassigned_ids:
    vdata = unassigned_daily[unassigned_daily["vehicle_id"].astype(str) == vid]
    if len(vdata) == 0:
        continue
    total_km = vdata["dist_sum_m"].sum() / 1000
    total_hr = vdata["duration_hr"].sum()
    lat_range = f"{vdata['lat_min'].min():.3f}-{vdata['lat_max'].max():.3f}"
    lon_range = f"{vdata['lon_min'].min():.3f}-{vdata['lon_max'].max():.3f}"
    anomaly_days = vdata[vdata["dist_sum_m"] > vdata["dist_sum_m"].quantile(0.75) * 1.5]
    print(f"\n车{vid}: {len(vdata)} 天 | 总里程: {total_km:.0f} km | 总时长: {total_hr:.0f} h")
    print(f"  纬度范围: {lat_range} | 经度范围: {lon_range}")
    if len(anomaly_days) > 0:
        days_str = ", ".join(d.strftime("%m/%d") for d in anomaly_days["date"])
        print(f"  ⚠️ 异常高里程日: {days_str}")

# POK 评分
unassigned_hotspots = hotspots[hotspots["vehicle_id"].astype(str).isin(unassigned_ids)]
pok_scores = {}
for _, row in unassigned_hotspots.iterrows():
    coord_key = f"({row['mean_lat']:.3f}, {row['mean_lon']:.3f})"
    w = row.get("weight", row.get("duration_min", 1))
    pok_scores[coord_key] = pok_scores.get(coord_key, 0) + w

if pok_scores:
    max_pok = max(pok_scores.values())
    pok_scores = {k: v / max_pok * 20 for k, v in pok_scores.items()}

# Top 3 POK 坐标
pok_sorted = sorted(pok_scores.items(), key=lambda x: x[1], reverse=True)
print(f"\nPOK 监视 Top 3 坐标聚类:")
for coord, score in pok_sorted[:3]:
    print(f"  {coord}: 评分 {score:.1f}")

## 4. Vector D: 网络裂隙分析

识别非工作时段跨部门共现与异常社交模式。

In [ ]:
# D: 网络裂隙
print("=" * 60)
print("Vector D: 网络裂隙")
print("=" * 60)

after_hours_edges = net_edges[(net_edges["after_hours"] > 0) | (net_edges["night"] > 0)]
cross_dept_ah = after_hours_edges[after_hours_edges["dept_a"] != after_hours_edges["dept_b"]]

print(f"非工作时段共现边: {len(after_hours_edges)} 条")
print(f"跨部门非工作时段边: {len(cross_dept_ah)} 条")

# 夜间跨部门 — 高度可疑
night_cross = after_hours_edges[(after_hours_edges["night"] > 0) & (after_hours_edges["dept_a"] != after_hours_edges["dept_b"])]
print(f"\n🔴 深夜+跨部门共现 (最高可疑):")
for _, row in night_cross.iterrows():
    print(f"  {row['employee_a']} ({row['dept_a']}) ↔ {row['employee_b']} ({row['dept_b']}) | "
          f"深夜: {int(row['night'])}次 | 非工作时段: {int(row['after_hours'])}次")

# 从共现坐标聚合
network_scores = {}
ah_cooc = cooc[cooc["is_after_hours"] == True]
for _, row in ah_cooc.iterrows():
    coord_key = f"({row['mean_lat']:.3f}, {row['mean_lon']:.3f})"
    base = 2 if row["is_night"] else 1
    network_scores[coord_key] = network_scores.get(coord_key, 0) + base

print(f"\n网络裂隙坐标评分: {len(network_scores)} 个")

## 5. 综合评分与可疑地点排序

综合 Vector A-D 的评分，计算每个候选地点的综合可疑评分并排序。

In [ ]:
# 综合评分
all_locations = set()
all_locations.update(econ_scores.keys())
all_locations.update(temporal_scores.keys())
all_locations.update(pok_scores.keys())
all_locations.update(network_scores.keys())

composite = []
for loc in all_locations:
    s_econ = econ_scores.get(loc, 0)
    s_temp = temporal_scores.get(loc, 0)
    s_pok  = pok_scores.get(loc, 0)
    s_net  = network_scores.get(loc, 0)
    composite.append({
        "location": loc, "economic": s_econ, "temporal": s_temp,
        "pok_surveillance": s_pok, "network_cleavage": s_net,
        "total_score": s_econ + s_temp + s_pok + s_net
    })

df_scores = pd.DataFrame(composite).sort_values("total_score", ascending=False)
named_locations = df_scores[~df_scores["location"].str.startswith("(")]

print("\n" + "=" * 70)
print("  Top 10 可疑活动地点 — 综合评分")
print("=" * 70)
for i, (_, row) in enumerate(named_locations.head(10).iterrows()):
    cat = loc_cat[loc_cat["location_clean"] == row["location"]]["location_category"].values
    cat_str = cat[0] if len(cat) > 0 else "?"
    print(f"  {i+1:2d}. {row['location']:<35s} [{cat_str:<12s}] "
          f"得分={row['total_score']:.1f} "
          f"(E:{row['economic']:.0f} T:{row['temporal']:.0f} "
          f"P:{row['pok_surveillance']:.0f} N:{row['network_cleavage']:.0f})")

# 保存评分表
named_locations.head(10).to_csv(DATA_DIR / "q5_suspicious_location_scores.csv", index=False)
print(f"\n✅ 评分表已保存至 data/processed/q5_suspicious_location_scores.csv")

## 6. 可视化 — 10 张证据图表

生成 10 张高信息密度图表，覆盖综合评分、雷达图、时间线、GPS 分布、经济异常和网络裂隙。

In [ ]:
# 运行完整分析脚本生成所有 10 张图表
%run ../scripts/run_q5_analysis.py

print("\n✅ 全部 10 张 Q5 图表已生成至 reports/figures/")

## 7. Q5 可疑地点证据链

### 综合评分 Top 10

| 排名 | 地点 | 类别 | 综合评分 |
|------|------|------|----------|
| 1 | Nationwide Refinery | Industrial | 104.7 |
| 2 | Abila Airport | Transport | 83.9 |
| 3 | Carlyle Chemical Inc. | Industrial | 83.7 |
| 4 | Stewart and Sons Fabrication | Industrial | 83.6 |
| 5 | Maximum Iron and Steel | Industrial | 35.2 |
| 6 | Kronos Pipe and Irrigation | Industrial | 29.1 |
| 7 | Abila Scrapyard | Industrial | 28.9 |
| 8 | Frydos Autosupply n' More | Industrial | 18.0 |
| 9 | Kronos Mart | Retail | 9.1 |
| 10 | Albert's Fine Clothing | Retail | 8.7 |

### 关键证据链

1. **Nationwide Refinery** — 最高经济异常评分，33 笔工业高价交易，总消费 $88,289，涉及多张 Corporate 卡片
2. **Abila Airport** — 最高总消费地点 ($137,358)，39 笔异常交易，交通枢纽特殊性
3. **Carlyle Chemical Inc.** — $86,940 总消费，26 笔异常，是唯一有 exact_noon 工业交易的场所
4. **Frydos Autosupply n' More** — **$10,000 极端交易** (CC_9551, 归属 Nils Calixto/Engineering)，金额精确为整数
5. **Kronos Mart** — **5 笔凌晨 3:00 交易**，100% 时间异常，涉及 5 张不同卡片

### 夜间 GPS 共现事件
- (36.074, 24.873): Bertrand Ovan (Facilities) ↔ Willem Vasco-Pais (Executive), 1/6 2AM
- (36.076, 24.871): Hennie Osvaldo (Security) ↔ Orhan Strum (Executive), 1/8 0AM
- (36.080, 24.872): Elsa Orilla (Engineering) ↔ Bertrand Ovan (Facilities), 1/14 3AM

### CC-Loyalty 数据篡改证据
- 1,307 对匹配中 226 对存在系统性价格偏移 (17%)
- $20/$60/$80 三种固定偏移模式，CC 价格始终高于 Loyalty
- 强烈暗示信用卡交易数据被人为抬高

## 8. 结论与建议

**最高优先级调查对象:**
1. **Nils Calixto (Engineering)** — 持有 11 张 Corporate 卡片，关联 Frydos Autosupply $10,000 交易
2. **未分配车辆 101/104/105/106/107** — 系统性地伴随员工车辆活动，尤其在 1/16 异常日
3. **Bertrand Ovan (Facilities)** — 涉及 2/3 的夜间 GPS 共现事件

**建议执法行动:**
- 冻结 CC_9551、CC_9220、CC_7792、CC_3506、CC_8642 等 Corporate 卡片
- 追踪未分配车辆 101-107 的注册信息
- 对 Nationwide Refinery、Carlyle Chemical 进行现场审计
- 对 Kronos Mart 凌晨交易进行监控录像核查